## Hypotheses

### Retention Day 1
- **H0:** The 1-day retention rate is the same between the gate_30 and gate_40 groups.
- **H1:** The 1-day retention rate differs between the gate_30 and gate_40 groups.

### Retention Day 7
- **H0:** The 7-day retention rate is the same between the gate_30 and gate_40 groups.
- **H1:** The 7-day retention rate differs between the gate_30 and gate_40 groups.

## Significance Level
- α = 0.05 (two-sided test)

## Minimum Detectable Effect (MDE)
A difference of **5 percentage points or more** is considered meaningful for a product decision.
Smaller differences, even if statistically significant, would not justify changing the gate.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportions_ztest
df = pd.read_csv("../data/cookie_cats_cleaned.csv")
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [4]:


def two_proportion_z_test(df, group_col, metric_col, group_a, group_b):
    a = df[df[group_col] == group_a][metric_col]
    b = df[df[group_col] == group_b][metric_col]
    
    count = np.array([a.sum(), b.sum()])
    nobs = np.array([len(a), len(b)])
    
    from statsmodels.stats.proportion import proportions_ztest
    z_stat, p_val = proportions_ztest(count, nobs)
    
    prop_a = count[0] / nobs[0]
    prop_b = count[1] / nobs[1]
    diff = prop_b - prop_a
    
    return {
        'group_a_rate': prop_a,
        'group_b_rate': prop_b,
        'difference': diff,
        'z_stat': z_stat,
        'p_value': p_val
    }

result_ret1 = two_proportion_z_test(df, 'version', 'retention_1', 'gate_30', 'gate_40')
print(result_ret1)

{'group_a_rate': np.float64(0.4481979462627799), 'group_b_rate': np.float64(0.44228274967574577), 'difference': np.float64(-0.005915196587034155), 'z_stat': np.float64(1.787103509763628), 'p_value': np.float64(0.0739207603418346)}


In [5]:
result_ret7 = two_proportion_z_test(df, 'version', 'retention_7', 'gate_30', 'gate_40')
print(result_ret7)

{'group_a_rate': np.float64(0.19018322557551623), 'group_b_rate': np.float64(0.18200004396667327), 'difference': np.float64(-0.00818318160884296), 'z_stat': np.float64(3.1574100858819936), 'p_value': np.float64(0.0015917731773993442)}


In [6]:
from statsmodels.stats.proportion import confint_proportions_2indep

def diff_confidence_interval(df, group_col, metric_col, group_a, group_b):
    a = df[df[group_col] == group_a][metric_col]
    b = df[df[group_col] == group_b][metric_col]
    
    count_a, nobs_a = a.sum(), len(a)
    count_b, nobs_b = b.sum(), len(b)
    
    ci_low, ci_upp = confint_proportions_2indep(
        count_a, nobs_a, count_b, nobs_b, method='wald'
    )
    return ci_low, ci_upp

ci_ret1 = diff_confidence_interval(df, 'version', 'retention_1', 'gate_40', 'gate_30')
ci_ret7 = diff_confidence_interval(df, 'version', 'retention_7', 'gate_40', 'gate_30')

print(f"retention_1 diff 95% CI: {ci_ret1}")
print(f"retention_7 diff 95% CI: {ci_ret7}")

retention_1 diff 95% CI: (np.float64(-0.012402509778563666), np.float64(0.0005721166044953558))
retention_7 diff 95% CI: (np.float64(-0.013263369909537494), np.float64(-0.0031029933081484252))


In [7]:
def bootstrap_diff(df, group_col, metric_col, group_a, group_b, n_iterations=10000):
    a = df[df[group_col] == group_a][metric_col].values
    b = df[df[group_col] == group_b][metric_col].values
    
    diffs = []
    for _ in range(n_iterations):
        sample_a = np.random.choice(a, size=len(a), replace=True)
        sample_b = np.random.choice(b, size=len(b), replace=True)
        diffs.append(sample_b.mean() - sample_a.mean())
    
    diffs = np.array(diffs)
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    return diffs, ci_low, ci_high

boot_diffs_ret1, boot_ci_low1, boot_ci_high1 = bootstrap_diff(
    df, 'version', 'retention_1', 'gate_30', 'gate_40'
)
print(f"Bootstrap 95% CI (retention_1 diff): [{boot_ci_low1:.4f}, {boot_ci_high1:.4f}]")

boot_diffs_ret7, boot_ci_low7, boot_ci_high7 = bootstrap_diff(
    df, 'version', 'retention_7', 'gate_30', 'gate_40'
)
print(f"Bootstrap 95% CI (retention_7 diff): [{boot_ci_low7:.4f}, {boot_ci_high7:.4f}]")

Bootstrap 95% CI (retention_1 diff): [-0.0123, 0.0005]
Bootstrap 95% CI (retention_7 diff): [-0.0133, -0.0032]


In [8]:
df['engagement_segment'] = pd.qcut(
    df['sum_gamerounds'], q=4, labels=['low', 'medium', 'high', 'very_high']
)

segment_results = {}
for segment in df['engagement_segment'].cat.categories:
    seg_df = df[df['engagement_segment'] == segment]
    res = two_proportion_z_test(seg_df, 'version', 'retention_7', 'gate_30', 'gate_40')
    segment_results[segment] = res

for seg, res in segment_results.items():
    print(seg, res)

low {'group_a_rate': np.float64(0.012193098871930989), 'group_b_rate': np.float64(0.013111128662822841), 'difference': np.float64(0.0009180297908918528), 'z_stat': np.float64(-0.6451899599995229), 'p_value': np.float64(0.518804091696032)}
medium {'group_a_rate': np.float64(0.041728468308118265), 'group_b_rate': np.float64(0.04093737845196422), 'difference': np.float64(-0.0007910898561540453), 'z_stat': np.float64(0.2837908629186871), 'p_value': np.float64(0.7765706571133868)}
high {'group_a_rate': np.float64(0.14772530704030445), 'group_b_rate': np.float64(0.1321880650994575), 'difference': np.float64(-0.015537241940846963), 'z_stat': np.float64(3.365283932280231), 'p_value': np.float64(0.0007646492903557162)}
very_high {'group_a_rate': np.float64(0.5674690007293947), 'group_b_rate': np.float64(0.54249390456287), 'difference': np.float64(-0.024975096166524602), 'z_stat': np.float64(3.763864686267442), 'p_value': np.float64(0.00016730743866314525)}
